# core

> Sanskrit recitation from text with Vāgdhenu: a chant voice for verse, a prose voice for gadya, and line timings.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Vāgdhenu ([code](https://github.com/prathoshap/vagdhenu), [weights](https://huggingface.co/prathoshap/vagdhenu), Apache-2.0) is an IndicF5 fine-tune
with a BigVGAN-v2 vocoder, trained on one speaker's chant. Neither its source nor BigVGAN's is a pip package, so both are fetched at pinned commits into
the cache on first use. The weights come from the Hub.

In [ ]:
#| export
import os, re, sys, io, tarfile, types, subprocess, urllib.request, numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from fastcore.utils import *

In [ ]:
#| export
SRCS = dict(vagdhenu=('prathoshap/vagdhenu', 'c18927a8a77775753e32248177a5a6bea1019a12', 'src'),
            bigvgan=('NVIDIA/BigVGAN', '7d2b454564a6c7d014227f635b7423881f14bdac', ''))
HF = 'prathoshap/vagdhenu'
VOICES = dict(steer='voice_steer_ema_2026-06-17.pt', armA='voice_armA_ema_2026-06-11.pt')
VOC, SR = 'voc_bigvgan_EMA_2026-06-11.pth', 24000
cache = Path(os.environ.get('VACHAKA_CACHE', '~/.cache/vachaka')).expanduser()

In [ ]:
#| export
def fetch_src(name):
    "Path to `name`'s source at its pinned commit, downloading the tarball once."
    repo, sha, sub = SRCS[name]
    d = cache/f'{name}-{sha[:10]}'
    if not d.exists():
        data = urllib.request.urlopen(f'https://codeload.github.com/{repo}/tar.gz/{sha}').read()
        with tarfile.open(fileobj=io.BytesIO(data)) as t: t.extractall(d, filter='data')
    return next(d.iterdir())/sub

In [ ]:
#| export
def setup_paths():
    "Put Vāgdhenu's `src` and BigVGAN on `sys.path`."
    for n in SRCS:
        p = str(fetch_src(n))
        if p not in sys.path: sys.path.insert(0, p)

In [ ]:
#| export
def best_device():
    "cuda, then mps, then cpu."
    import torch
    if torch.cuda.is_available(): return 'cuda'
    return 'mps' if torch.backends.mps.is_available() else 'cpu'

## Text

The model reads plain Devanagari. Vedic accents, digits, and verse numbers between daṇḍas are dropped. Vāgdhenu has no svaras and is for laukika chant.

In [ ]:
#| export
_ACC = re.compile('[\u0951-\u0954\u1CD0-\u1CFF\uA8E0-\uA8FF]')
def clean(s):
    "Plain Devanagari for the model: no accents, digits or numbered daṇḍas."
    s = _ACC.sub('', s)
    s = re.sub('[0-9\u0966-\u096F]+', '', s)
    s = re.sub('[।॥|]+(\\s*[।॥|]+)*', lambda m: '॥' if '॥' in m[0] else '।', s)
    return re.sub('\\s+', ' ', s).strip()

In [ ]:
test_eq(clean('अग्निमी॑ळे पु॒रोहि॑तं ॥ १ ॥'), 'अग्निमीळे पुरोहितं ॥')
test_eq(clean('धर्मक्षेत्रे कुरुक्षेत्रे ।'), 'धर्मक्षेत्रे कुरुक्षेत्रे ।')

In [ ]:
#| export
def n_syll(s):
    "Akṣaras in Devanagari `s`: independent vowels and consonants not killed by a virāma."
    return sum(1 for i,c in enumerate(s) if '\u0905'<=c<='\u0914' or ('\u0915'<=c<='\u0939' and s[i+1:i+2]!='्'))

In [ ]:
test_eq(n_syll('कर्मण्येवाधिकारस्ते'), 8)

In [ ]:
#| export
def chunk_prose(text, max_syll=24):
    "Split prose at daṇḍas, then at word breaks, into pieces of at most `max_syll` akṣaras."
    out = []
    for sent in re.split('[।॥|]', clean(text)):
        cur, cs = [], 0
        for w in sent.split():
            ws = n_syll(w)
            if cur and cs+ws > max_syll: out.append(' '.join(cur)); cur, cs = [], 0
            cur.append(w); cs += ws
        if cur: out.append(' '.join(cur))
    return out

In [ ]:
p = chunk_prose('अथातो धर्मजिज्ञासा । चोदनालक्षणोऽर्थो धर्मः । तस्य निमित्तपरीष्टिः ॥')
test_eq(p, ['अथातो धर्मजिज्ञासा', 'चोदनालक्षणोऽर्थो धर्मः', 'तस्य निमित्तपरीष्टिः'])
assert all(n_syll(c) <= 24 for c in chunk_prose('क '*80))

## Recitation

The rendered audio with one `(start_ms, end_ms)` span per piece. Each piece is a line given to `Chanter.chant`, or a chunk from `Chanter.read`.

In [ ]:
#| export
@dataclass
class Recitation:
    audio: np.ndarray
    spans: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    sr: int = SR
    cer: float = None
    @property
    def ms(self): return round(len(self.audio)*1000/self.sr)
    def __add__(self, o): return concat([self, o])
    def save(self, path, kbps=96):
        "Write a wav, or an mp3 through ffmpeg when `path` ends in .mp3."
        import soundfile as sf
        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
        if path.suffix != '.mp3': sf.write(path, self.audio, self.sr); return path
        buf = io.BytesIO(); sf.write(buf, self.audio, self.sr, format='WAV')
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'wav', '-i', '-', '-b:a', f'{kbps}k', str(path)], input=buf.getvalue(), check=True)
        return path

In [ ]:
#| export
def concat(recs, gap=0.8):
    "Join recitations with `gap` seconds of silence, shifting each one's spans."
    recs = L(recs); sr = recs[0].sr; sil = np.zeros(int(gap*sr), np.float32)
    aus, spans, texts, pos = [], [], [], 0
    for i,r in enumerate(recs):
        if i: aus.append(sil); pos += len(sil)
        off = round(pos*1000/sr)
        spans += [(s+off, e+off) for s,e in r.spans]; texts += r.texts
        aus.append(r.audio); pos += len(r.audio)
    return Recitation(np.concatenate(aus), spans, texts, sr)

In [ ]:
a = Recitation(np.zeros(SR, np.float32), [(0, 900)], ['क'])
b = a + a
test_eq(b.ms, 2800)
test_eq(b.spans, [(0, 900), (1800, 2700)])

## Chanter

Loads the DiT, the vocoder and the reference bank once. On Apple silicon the DiT runs on MPS and BigVGAN on the CPU, since MPS lacks a
transposed convolution with more than 65,536 output channels. `chant` renders verse against the bank's reference for its metre; `read` renders prose
against the gadya reference by default. The reference sets pace and melody; the timbre is the fine-tuned voice.

In [ ]:
#| export
def _stitch(self, segs, gaps, fric=False, halant=False):
    "Render-core's stitch, also recording each gated piece's span in samples."
    from render_core import gate
    last, out, self.spans, pos = len(segs)-1, [], [], 0
    for i,s in enumerate(segs):
        g = gate(s, fric=fric and i==0, halant=halant and i==last)
        self.spans.append((pos, pos+len(g))); pos += len(g); out.append(g)
        if i < last: gp = gaps[i] if i < len(gaps) else gaps[-1]; out.append(gp); pos += len(gp)
    return np.concatenate(out)

In [ ]:
#| export
def _cpu_vocoder(r):
    "Move BigVGAN to the CPU."
    import torch
    r.g = r.g.to('cpu')
    def bv(self, mel):
        m = torch.from_numpy(mel)
        with torch.no_grad():
            if m.dim() == 3 and m.shape[1] != 100 and m.shape[2] == 100: m = m.transpose(1, 2)
            return self.g(m).squeeze().numpy().astype(np.float32)
    r._bvgan = types.MethodType(bv, r)

In [ ]:
#| export
class Chanter:
    "Sanskrit verse and prose to audio with line spans."
    def __init__(self, voice='steer', device=None, nfe=64, speed=0.9, gap=0.55, prose_gap=0.3):
        from huggingface_hub import hf_hub_download
        setup_paths()
        from render_core import Renderer, detect_meter_key
        self.detect, device = detect_meter_key, ifnone(device, best_device())
        store_attr('voice,device,gap,prose_gap')
        dl = partial(hf_hub_download, HF)
        bank = fetch_src('vagdhenu')/'reference_bank'/'bank.json'
        self.r = Renderer(dl(VOICES[voice]), dl(VOC), str(bank), device=device, vocab_file=dl('vocab.txt'), nfe=nfe, speed=speed, gap=gap)
        self.r._stitch = types.MethodType(_stitch, self.r)
        if device == 'mps': _cpu_vocoder(self.r)

    def meters(self): return self.r.meters()
    def meter(self, lines): return self.detect(' । '.join(lines)) or ''

    def _render(self, pieces, meter, seed, gap):
        self.r.gap = gap
        sr, au = self.r.render_one(pieces, meter, seed=seed)
        return Recitation(au, [(round(s*1000/sr), round(e*1000/sr)) for s,e in self.r.spans], pieces, sr)

    def chant(self, lines, meter=None, seed=60):
        "Chant one verse given as its lines (a str splits at newlines and daṇḍas); one span per line."
        if isinstance(lines, str): lines = re.split('[\n।॥|]', lines)
        lines = [o for o in (clean(l).strip(' ।॥') for l in lines) if o]
        return self._render(lines, ifnone(meter, self.meter(lines)) or 'vasantatilaka', seed, self.gap)

    def read(self, text, ref='gadya', seed=60, max_syll=24):
        "Read prose against the `ref` reference (gadya, or a metre's); one span per chunk."
        return self._render(chunk_prose(text, max_syll), ref, seed, self.prose_gap)

Rendering needs the weights (about 1.5 GB on first use) and takes several times real time on Apple silicon, so these cells do not run in the tests.

In [ ]:
#| eval: false
ch = Chanter()
r = ch.chant(['वसुदेवसुतं देवं कंसचाणूरमर्दनम्', 'देवकीपरमानन्दं कृष्णं वन्दे जगद्गुरुम्'])
r.spans, r.save('/tmp/krishna.mp3')

In [ ]:
#| eval: false
ch.read('अथातो धर्मजिज्ञासा । चोदनालक्षणोऽर्थो धर्मः ।').save('/tmp/mimamsa.mp3')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()